# Projeto Final – Analista de Dados (EBAC)
## Acidentes nas rodovias federais de Pernambuco (PRF, 2025)
### Etapa 3 – Tratamento e Preparação dos Dados

Com base nos problemas encontrados na Etapa 2, esta etapa:
1. **Trata valores ausentes e inconsistências** (duplicatas, "Ignorado", idades e coordenadas impossíveis, grafia de municípios)
2. **Padroniza formatos** (datas, horas, categorias de texto, números)
3. **Cria colunas derivadas** (mês, faixa horária, fim de semana, gravidade, faixa etária, grupo de veículo, idade do veículo, trecho, taxa por 100 mil hab., etc.)
4. **Entrega tabelas prontas** para o Looker Studio

**Entrada:** `pe_ocorrencias.csv`, `pe_pessoas.csv`, `pe_populacao.csv` (gerados na Etapa 1).
**Saída:** `final_ocorrencias.csv`, `final_pessoas.csv`, `final_municipios.csv`.

In [1]:
import re, unicodedata
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 60)

oc  = pd.read_csv('pe_ocorrencias.csv', low_memory=False)
pes = pd.read_csv('pe_pessoas.csv', low_memory=False)
pop = pd.read_csv('pe_populacao.csv')

# Guardamos o "antes" para comparar no final
antes = {'ocorrencias': oc.copy(), 'pessoas': pes.copy(), 'municipios': pop.copy()}
log = []   # registro de tudo que foi tratado

def registrar(tabela, acao, linhas_afetadas):
    log.append({'tabela': tabela, 'tratamento': acao, 'linhas_afetadas': int(linhas_afetadas)})

print(f'Ocorrências: {len(oc):,} | Pessoas: {len(pes):,} | Municípios: {len(pop):,}')

Ocorrências: 3,013 | Pessoas: 7,646 | Municípios: 185


## 1. Funções de padronização

In [2]:
SEM_INFO = {'IGNORADO', 'IGNORADA', 'NAO INFORMADO', 'NAO INFORMADA', '(NULL)', 'NULL', 'NAN', 'NONE', ''}

def sem_acento(s):
    return unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode()

def chave(s):
    # Nome de município padronizado para cruzamento: sem acento, maiúsculo, sem pontuação
    return re.sub(r'\s+', ' ', re.sub(r'[^A-Za-z ]', ' ', sem_acento(s))).strip().upper()

def padronizar_texto(serie, capitalizar=True):
    # Remove espaços extras, deixa 'Primeira letra maiúscula' e unifica os valores sem informação
    s = serie.astype(object).where(serie.notna(), '').astype(str).str.strip().str.replace(r'\s+', ' ', regex=True)
    vazio = s.map(lambda x: sem_acento(x).upper()).isin(SEM_INFO)
    if capitalizar:
        s = s.str.capitalize()
    s[vazio] = 'Não informado'
    return s, int(vazio.sum())

## 2. Tabela `ocorrencias`

### 2.1 Duplicatas

In [3]:
n = oc.duplicated('id').sum()
oc = oc.drop_duplicates('id')
registrar('ocorrencias', 'Remoção de ocorrências duplicadas (id)', n)
print('Duplicatas removidas:', n)

Duplicatas removidas: 0


### 2.2 Padronização de datas e horas

In [4]:
data_iso = pd.to_datetime(oc['data_inversa'], format='%Y-%m-%d', errors='coerce')
data_br  = pd.to_datetime(oc['data_inversa'], format='%d/%m/%Y', errors='coerce')
oc['data'] = data_iso.fillna(data_br)            # aceita os dois formatos que a PRF já usou

hora_dt = pd.to_datetime(oc['horario'], format='%H:%M:%S', errors='coerce')
oc['horario'] = hora_dt.dt.strftime('%H:%M:%S')

n_data_inv = oc['data'].isna().sum()
oc = oc.dropna(subset=['data'])
registrar('ocorrencias', 'Linhas com data inválida removidas', n_data_inv)
print('Datas inválidas removidas:', n_data_inv, '| Horários inválidos:', hora_dt.isna().sum())

Datas inválidas removidas: 0 | Horários inválidos: 0


### 2.3 Padronização de categorias (texto)

In [5]:
colunas_texto_oc = ['dia_semana','causa_acidente','tipo_acidente','classificacao_acidente','fase_dia',
                    'sentido_via','condicao_metereologica','tipo_pista','tracado_via','uso_solo',
                    'regional','delegacia','uop']
siglas = {'regional','delegacia','uop'}          # códigos: só tiramos espaços, sem mudar maiúsculas
for c in [c for c in colunas_texto_oc if c in oc.columns]:
    oc[c], n = padronizar_texto(oc[c], capitalizar=c not in siglas)
    if n: registrar('ocorrencias', f'"{c}": Ignorado/nulo -> "Não informado"', n)

# Dia da semana recalculado a partir da data (garante consistência)
dias = ['Segunda','Terça','Quarta','Quinta','Sexta','Sábado','Domingo']
oc['dia_semana'] = oc['data'].dt.dayofweek.map(dict(enumerate(dias)))
oc['dia_semana_num'] = oc['data'].dt.dayofweek + 1        # 1 = segunda (para ordenar no Looker)

oc['condicao_metereologica'].value_counts().head()

,count
condicao_metereologica,
Céu claro,2118
Nublado,353
Sol,223
Chuva,205
Garoa/chuvisco,74


### 2.4 Padronização numérica e valores ausentes

In [6]:
# Contagens de vítimas: ausente significa zero
contagens = ['pessoas','mortos','feridos_leves','feridos_graves','ilesos','ignorados','feridos','veiculos']
contagens = [c for c in contagens if c in oc.columns]
n = int(oc[contagens].isna().sum().sum())
oc[contagens] = oc[contagens].fillna(0).astype(int)
registrar('ocorrencias', 'Contagens de vítimas/veículos nulas -> 0', n)

# BR como número inteiro, km como decimal
oc['br'] = pd.to_numeric(oc['br'], errors='coerce').astype('Int64')
oc['km'] = pd.to_numeric(oc['km'].astype(str).str.replace(',', '.'), errors='coerce')
registrar('ocorrencias', 'BR ou km ausentes (mantidos como nulo)', oc['br'].isna().sum() + oc['km'].isna().sum())

# Coordenadas fora de Pernambuco viram nulo (não aparecem no mapa, mas o acidente continua contando)
for c in ['latitude','longitude']:
    oc[c] = pd.to_numeric(oc[c].astype(str).str.replace(',', '.'), errors='coerce')
fora = ~(oc['latitude'].between(-10, -7) & oc['longitude'].between(-42, -32))
oc.loc[fora, ['latitude','longitude']] = np.nan
registrar('ocorrencias', 'Coordenadas fora de PE -> nulo', fora.sum())
print('Coordenadas anuladas:', fora.sum())

Coordenadas anuladas: 0


### 2.5 Município: correção de grafia para cruzar com o IBGE

In [7]:
# Grafias da PRF que diferem do IBGE (encontradas na Etapa 1)
CORRECOES_MUNICIPIO = {
    'BELEM DE SAO FRANCISCO': 'BELEM DO SAO FRANCISCO',
}

oc['municipio_chave'] = oc['municipio'].map(chave).replace(CORRECOES_MUNICIPIO)
pop['municipio_chave'] = pop['municipio'].map(chave)

n = oc['municipio'].map(chave).isin(CORRECOES_MUNICIPIO.keys()).sum()
registrar('ocorrencias', 'Grafia de município corrigida para o padrão IBGE', n)

# Nome oficial (com acento) e código IBGE vindos da tabela de população
pop['cod_ibge'] = '26' + pop['cod_munic'].astype(str).str.split('.').str[0].str.zfill(5)
oc = oc.merge(pop[['municipio_chave','municipio','cod_ibge']].rename(columns={'municipio':'municipio_ibge'}),
              on='municipio_chave', how='left')
oc['municipio'] = oc['municipio_ibge'].fillna(oc['municipio'].str.title())
oc = oc.drop(columns='municipio_ibge')

print('Ocorrências sem correspondência no IBGE:', oc['cod_ibge'].isna().sum())

Ocorrências sem correspondência no IBGE: 0


### 2.6 Colunas derivadas

In [8]:
# Tempo
oc['ano']      = oc['data'].dt.year
oc['mes']      = oc['data'].dt.month
oc['mes_nome'] = oc['mes'].map(dict(enumerate(['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez'], 1)))
oc['ano_mes']  = oc['data'].dt.strftime('%Y-%m')
oc['hora']     = pd.to_datetime(oc['horario'], format='%H:%M:%S', errors='coerce').dt.hour.astype('Int64')
oc['faixa_horaria'] = pd.cut(oc['hora'], bins=[-1, 5, 11, 17, 23],
                             labels=['Madrugada (0-5h)','Manhã (6-11h)','Tarde (12-17h)','Noite (18-23h)'])
oc['faixa_horaria'] = oc['faixa_horaria'].cat.add_categories('Não informado').fillna('Não informado').astype(str)
oc['fim_de_semana'] = np.where(oc['dia_semana_num'] >= 6, 'Fim de semana', 'Dia útil')

# Gravidade do acidente (pior desfecho)
oc['gravidade'] = np.select(
    [oc['mortos'] > 0, oc['feridos_graves'] > 0, oc['feridos_leves'] > 0],
    ['Com mortos', 'Com feridos graves', 'Com feridos leves'], default='Sem vítimas')
oc['com_morte'] = (oc['mortos'] > 0).astype(int)

# Local
oc['br_nome'] = ('BR-' + oc['br'].astype(str)).where(oc['br'].notna(), 'Não informado')
faixa = (oc['km'] // 10 * 10).astype('Int64')
oc['trecho_10km'] = oc['br_nome'] + ' | km ' + faixa.astype(str) + '-' + (faixa + 10).astype(str)
oc.loc[oc['km'].isna() | oc['br'].isna(), 'trecho_10km'] = 'Não informado'
oc['lat_long'] = np.where(oc['latitude'].notna(),
                          oc['latitude'].round(6).astype(str) + ',' + oc['longitude'].round(6).astype(str), None)

registrar('ocorrencias', 'Colunas derivadas criadas: ano, mes, mes_nome, ano_mes, hora, faixa_horaria, '
          'fim_de_semana, dia_semana_num, gravidade, com_morte, br_nome, trecho_10km, lat_long, cod_ibge', len(oc))

oc[['data','horario','faixa_horaria','fim_de_semana','gravidade','trecho_10km','lat_long']].head()

,data,horario,faixa_horaria,fim_de_semana,gravidade,trecho_10km,lat_long
0,2025-01-01,17:00:00,Tarde (12-17h),Dia útil,Sem vítimas,BR-407 | km 0-10,"-8.475031,-41.01371"
1,2025-01-02,11:57:00,Manhã (6-11h),Dia útil,Com feridos leves,BR-232 | km 10-20,"-8.094142,-35.037166"
2,2025-01-02,13:01:00,Tarde (12-17h),Dia útil,Sem vítimas,BR-101 | km 40-50,"-7.87406,-34.905881"
3,2025-01-03,04:12:00,Madrugada (0-5h),Dia útil,Sem vítimas,BR-101 | km 20-30,"-7.729397,-34.941859"
4,2025-01-03,06:30:00,Manhã (6-11h),Dia útil,Com feridos graves,BR-232 | km 270-280,"-8.370731,-37.261263"


## 3. Tabela `pessoas`

In [9]:
# 3.1 Duplicatas e vínculo com ocorrências
n = pes.duplicated('pesid').sum()
pes = pes.drop_duplicates('pesid')
registrar('pessoas', 'Remoção de pessoas duplicadas (pesid)', n)

sem_ocorrencia = ~pes['id'].isin(oc['id'])
registrar('pessoas', 'Pessoas sem ocorrência correspondente removidas', sem_ocorrencia.sum())
pes = pes[~sem_ocorrencia].copy()

# 3.2 Categorias
for c in [c for c in ['tipo_veiculo','marca','tipo_envolvido','estado_fisico','sexo'] if c in pes.columns]:
    pes[c], n = padronizar_texto(pes[c], capitalizar=c != 'marca')
    if n: registrar('pessoas', f'"{c}": Ignorado/nulo -> "Não informado"', n)

pes['sexo'] = pes['sexo'].replace({'M': 'Masculino', 'F': 'Feminino'})
pes['estado_fisico'] = pes['estado_fisico'].replace({'Obito': 'Óbito', 'Morto': 'Óbito'})
print(pes['estado_fisico'].value_counts(), '\n')
print(pes['sexo'].value_counts())

estado_fisico
Ileso            2938
Lesões leves     2254
Lesões graves    1094
Não informado     678
Óbito             336
Name: count, dtype: int64 

sexo
Masculino        5186
Feminino         1427
Não informado     687
Name: count, dtype: int64


In [10]:
# 3.3 Idade: valores impossíveis viram nulo (não inventamos idade)
pes['idade'] = pd.to_numeric(pes['idade'], errors='coerce')
invalida = pes['idade'].notna() & ~pes['idade'].between(0, 100)
registrar('pessoas', 'Idade < 0 ou > 100 -> nulo', invalida.sum())
pes.loc[invalida, 'idade'] = np.nan
pes['idade'] = pes['idade'].astype('Int64')

# 3.4 Ano de fabricação do veículo
if 'ano_fabricacao_veiculo' in pes.columns:
    pes['ano_fabricacao_veiculo'] = pd.to_numeric(pes['ano_fabricacao_veiculo'], errors='coerce')
    inval = pes['ano_fabricacao_veiculo'].notna() & ~pes['ano_fabricacao_veiculo'].between(1950, 2026)
    registrar('pessoas', 'Ano de fabricação fora de 1950-2026 -> nulo', inval.sum())
    pes.loc[inval, 'ano_fabricacao_veiculo'] = np.nan
    pes['ano_fabricacao_veiculo'] = pes['ano_fabricacao_veiculo'].astype('Int64')

print('Idades nulas após tratamento:', pes['idade'].isna().sum())

Idades nulas após tratamento: 4


In [11]:
# 3.5 Colunas derivadas
pes['faixa_etaria'] = pd.cut(pes['idade'].astype('float'), bins=[-1, 17, 24, 34, 44, 59, 100],
                             labels=['0-17','18-24','25-34','35-44','45-59','60+'])
pes['faixa_etaria'] = pes['faixa_etaria'].cat.add_categories('Não informado').fillna('Não informado').astype(str)

def grupo_veiculo(v):
    v = sem_acento(v).lower()
    if any(k in v for k in ['motocicleta', 'motoneta', 'ciclomotor', 'triciclo', 'quadriciclo']): return 'Moto'
    if any(k in v for k in ['automovel', 'camioneta', 'caminhonete', 'utilitario']):             return 'Carro/Utilitário'
    if any(k in v for k in ['caminhao', 'reboque', 'semi', 'trator']):                            return 'Caminhão'
    if any(k in v for k in ['onibus', 'micro']):                                                  return 'Ônibus/Micro-ônibus'
    if 'bicicleta' in v:                                                                           return 'Bicicleta'
    if 'nao informado' in v or v in ('', 'nan'):                                                   return 'Não informado'
    return 'Outros'

pes['grupo_veiculo'] = pes['tipo_veiculo'].map(grupo_veiculo)
pes.loc[pes['tipo_envolvido'].str.contains('Pedestre', na=False), 'grupo_veiculo'] = 'Pedestre'

if 'ano_fabricacao_veiculo' in pes.columns:
    pes['idade_veiculo'] = (2025 - pes['ano_fabricacao_veiculo']).astype('Int64')

pes['vitima_fatal'] = (pes['estado_fisico'] == 'Óbito').astype(int)
pes['ferido']       = pes['estado_fisico'].str.contains('Lesões', na=False).astype(int)

registrar('pessoas', 'Colunas derivadas criadas: faixa_etaria, grupo_veiculo, idade_veiculo, vitima_fatal, ferido', len(pes))
pd.crosstab(pes['grupo_veiculo'], pes['estado_fisico'], margins=True)

estado_fisico,Ileso,Lesões graves,Lesões leves,Não informado,Óbito,All
grupo_veiculo,,,,,,
Bicicleta,3,27,36,2,13,81
Caminhão,463,51,114,109,17,754
Carro/Utilitário,1996,251,756,314,62,3379
Moto,164,639,1221,116,164,2304
Não informado,0,0,4,84,2,90
Outros,2,2,2,28,2,36
Pedestre,3,93,49,12,58,215
Ônibus/Micro-ônibus,307,31,72,13,18,441
All,2938,1094,2254,678,336,7300


In [12]:
# 3.6 Mantemos em 'pessoas' só o que é da pessoa + chaves; dados do acidente vêm do cruzamento por 'id'
colunas_pessoa = ['id','pesid','id_veiculo','tipo_veiculo','grupo_veiculo','marca','ano_fabricacao_veiculo',
                  'idade_veiculo','tipo_envolvido','estado_fisico','vitima_fatal','ferido','idade','faixa_etaria','sexo']
pes = pes[[c for c in colunas_pessoa if c in pes.columns]]

# Algumas colunas do acidente úteis para filtros no dashboard
pes = pes.merge(oc[['id','data','mes','mes_nome','municipio','br_nome','tipo_acidente','causa_acidente']], on='id', how='left')
pes.head()

,id,pesid,id_veiculo,tipo_veiculo,grupo_veiculo,marca,ano_fabricacao_veiculo,idade_veiculo,tipo_envolvido,estado_fisico,vitima_fatal,ferido,idade,faixa_etaria,sexo,data,mes,mes_nome,municipio,br_nome,tipo_acidente,causa_acidente
0,652648,1468101,1173253,Caminhão,Caminhão,DAF/CF FAC 310,2024,1,Passageiro,Ileso,0,0,57,45-59,Masculino,2025-01-01,1,Jan,Afrânio,BR-407,Incêndio,Demais falhas mecânicas ou elétricas
1,652648,1468100,1173253,Caminhão,Caminhão,DAF/CF FAC 310,2024,1,Condutor,Ileso,0,0,61,60+,Masculino,2025-01-01,1,Jan,Afrânio,BR-407,Incêndio,Demais falhas mecânicas ou elétricas
2,652750,1468357,1173450,Caminhão,Caminhão,SCANIA/P 310 B8X2,2013,12,Condutor,Ileso,0,0,25,25-34,Masculino,2025-01-02,1,Jan,Jaboatão dos Guararapes,BR-232,Colisão traseira,Acessar a via sem observar a presença dos outros veículos
3,652762,1468316,1173423,Caminhão,Caminhão,VW/10.160 DRC 4X2,2017,8,Condutor,Ileso,0,0,26,25-34,Masculino,2025-01-02,1,Jan,Igarassu,BR-101,Colisão traseira,Reação tardia ou ineficiente do condutor
4,652867,1468659,1173660,Caminhão,Caminhão,VW/17.260 CRM 4X2 4P,2020,5,Condutor,Ileso,0,0,34,25-34,Masculino,2025-01-03,1,Jan,Igarassu,BR-101,Colisão traseira,Manobra de mudança de faixa


## 4. Tabela `municipios` (população + indicadores agregados)

In [13]:
agg = oc.groupby('cod_ibge').agg(acidentes=('id','count'), mortos=('mortos','sum'),
                                 feridos=('feridos','sum'), acidentes_com_morte=('com_morte','sum'))

mun = pop[['cod_ibge','municipio','populacao']].merge(agg, on='cod_ibge', how='left')
cols = ['acidentes','mortos','feridos','acidentes_com_morte']
mun[cols] = mun[cols].fillna(0).astype(int)          # município sem acidente em BR federal = 0

# Colunas derivadas
mun['acidentes_100mil_hab'] = (mun['acidentes'] / mun['populacao'] * 100_000).round(2)
mun['mortos_100mil_hab']    = (mun['mortos'] / mun['populacao'] * 100_000).round(2)
mun['letalidade_pct']       = np.where(mun['acidentes'] > 0, (mun['mortos'] / mun['acidentes'] * 100).round(1), 0)
mun['porte'] = pd.cut(mun['populacao'], [0, 20_000, 100_000, 500_000, np.inf],
                      labels=['Pequeno (até 20 mil)','Médio (20-100 mil)','Grande (100-500 mil)','Metrópole (500 mil+)']).astype(str)
mun['municipio_uf'] = mun['municipio'] + ', PE, Brasil'    # ajuda o Looker a geolocalizar no mapa

registrar('municipios', 'Colunas derivadas criadas: cod_ibge, acidentes, mortos, acidentes_100mil_hab, '
          'mortos_100mil_hab, letalidade_pct, porte, municipio_uf', len(mun))
mun.sort_values('acidentes', ascending=False).head(10)

,cod_ibge,municipio,populacao,acidentes,mortos,feridos,acidentes_com_morte,acidentes_100mil_hab,mortos_100mil_hab,letalidade_pct,porte,municipio_uf
131,2611606,Recife,1588376,615,26,708,25,38.72,1.64,4.2,Metrópole (500 mil+),"Recife, PE, Brasil"
125,2611101,Petrolina,418444,280,14,306,13,66.91,3.35,5.0,Grande (100-500 mil),"Petrolina, PE, Brasil"
87,2607901,Jaboatão dos Guararapes,684293,218,11,213,11,31.86,1.61,5.0,Metrópole (500 mil+),"Jaboatão dos Guararapes, PE, Brasil"
44,2604106,Caruaru,405408,200,14,222,14,49.33,3.45,7.0,Grande (100-500 mil),"Caruaru, PE, Brasil"
74,2606804,Igarassu,123017,140,4,134,3,113.81,3.25,2.9,Grande (100-500 mil),"Igarassu, PE, Brasil"
66,2606002,Garanhuns,151803,90,10,93,9,59.29,6.59,11.1,Grande (100-500 mil),"Garanhuns, PE, Brasil"
157,2613909,Serra Talhada,98816,88,8,114,8,89.05,8.10,9.1,Médio (20-100 mil),"Serra Talhada, PE, Brasil"
0,2600054,Abreu e Lima,104248,86,4,102,3,82.50,3.84,4.7,Grande (100-500 mil),"Abreu e Lima, PE, Brasil"
70,2606408,Gravatá,92429,86,4,92,4,93.04,4.33,4.7,Médio (20-100 mil),"Gravatá, PE, Brasil"
137,2612208,Salgueiro,65946,58,6,73,5,87.95,9.10,10.3,Médio (20-100 mil),"Salgueiro, PE, Brasil"


## 5. Validação final

In [14]:
oc = oc.drop(columns=['data_inversa','uf','municipio_chave'], errors='ignore')
oc['data'] = oc['data'].dt.strftime('%Y-%m-%d')
pes['data'] = pd.to_datetime(pes['data']).dt.strftime('%Y-%m-%d')

checagens = {
    'ocorrencias: id único':                oc['id'].is_unique,
    'ocorrencias: sem data nula':           oc['data'].notna().all(),
    'ocorrencias: contagens sem nulo':      oc[contagens].notna().all().all(),
    'ocorrencias: todas com cod_ibge':      oc['cod_ibge'].notna().all(),
    'pessoas: pesid único':                 pes['pesid'].is_unique,
    'pessoas: todas ligadas a ocorrência':  pes['id'].isin(oc['id']).all(),
    'pessoas: idade entre 0 e 100 ou nula': pes['idade'].dropna().between(0, 100).all(),
    'municipios: 185 municípios':           len(mun) == 185,
    'municipios: soma de acidentes bate':   mun['acidentes'].sum() == oc['cod_ibge'].notna().sum(),
}
for k, v in checagens.items():
    print('✅' if v else '❌', k)

✅ ocorrencias: id único
✅ ocorrencias: sem data nula
✅ ocorrencias: contagens sem nulo
✅ ocorrencias: todas com cod_ibge
✅ pessoas: pesid único
✅ pessoas: todas ligadas a ocorrência
✅ pessoas: idade entre 0 e 100 ou nula
✅ municipios: 185 municípios
✅ municipios: soma de acidentes bate


In [15]:
# Antes x depois
comparativo = pd.DataFrame({
    'linhas antes':   [len(antes['ocorrencias']), len(antes['pessoas']), len(antes['municipios'])],
    'linhas depois':  [len(oc), len(pes), len(mun)],
    'colunas antes':  [antes['ocorrencias'].shape[1], antes['pessoas'].shape[1], antes['municipios'].shape[1]],
    'colunas depois': [oc.shape[1], pes.shape[1], mun.shape[1]],
    'nulos antes':    [antes['ocorrencias'].isna().sum().sum(), antes['pessoas'].isna().sum().sum(), antes['municipios'].isna().sum().sum()],
    'nulos depois':   [oc.isna().sum().sum(), pes.isna().sum().sum(), mun.isna().sum().sum()],
}, index=['ocorrencias','pessoas','municipios'])
display(comparativo)

print('\nRegistro de tratamentos:')
pd.DataFrame(log)

,linhas antes,linhas depois,colunas antes,colunas depois,nulos antes,nulos depois
ocorrencias,3013,3013,31,43,1,0
pessoas,7646,7300,35,22,1350,1818
municipios,185,185,5,12,0,0



Registro de tratamentos:


,tabela,tratamento,linhas_afetadas
0,ocorrencias,Remoção de ocorrências duplicadas (id),0
1,ocorrencias,Linhas com data inválida removidas,0
2,ocorrencias,"""sentido_via"": Ignorado/nulo -> ""Não informado""",9
3,ocorrencias,"""condicao_metereologica"": Ignorado/nulo -> ""Não informado""",32
4,ocorrencias,"""uop"": Ignorado/nulo -> ""Não informado""",1
5,ocorrencias,Contagens de vítimas/veículos nulas -> 0,0
6,ocorrencias,BR ou km ausentes (mantidos como nulo),0
7,ocorrencias,Coordenadas fora de PE -> nulo,0
8,ocorrencias,Grafia de município corrigida para o padrão IBGE,20
9,ocorrencias,"Colunas derivadas criadas: ano, mes, mes_nome, ano_mes, hora, faixa_horaria, fim_de_semana, dia_semana_num, gravidade, com_morte, br_nome, trecho_10km, lat_long, cod_ibge",3013


> Nulos que permanecem depois do tratamento são **intencionais**: idade/ano do veículo impossíveis e coordenadas fora de PE
> viram nulo em vez de um valor inventado. Para categorias, o ausente vira "Não informado", que aparece como opção nos filtros do dashboard.

## 6. Exportar para o Looker Studio

In [16]:
oc.to_csv('final_ocorrencias.csv', index=False, encoding='utf-8')
pes.to_csv('final_pessoas.csv', index=False, encoding='utf-8')
mun.to_csv('final_municipios.csv', index=False, encoding='utf-8')
pd.DataFrame(log).to_csv('log_tratamento.csv', index=False, encoding='utf-8')
print('Arquivos gerados: final_ocorrencias.csv, final_pessoas.csv, final_municipios.csv, log_tratamento.csv')

from google.colab import files
for f in ['final_ocorrencias.csv', 'final_pessoas.csv', 'final_municipios.csv', 'log_tratamento.csv']:
    files.download(f)

Arquivos gerados: final_ocorrencias.csv, final_pessoas.csv, final_municipios.csv, log_tratamento.csv


### Dicionário das colunas criadas
| Tabela | Coluna | Descrição |
|---|---|---|
| ocorrencias | `mes`, `mes_nome`, `ano_mes` | mês do acidente (número, nome e AAAA-MM) |
| ocorrencias | `hora`, `faixa_horaria` | hora do dia e faixa (madrugada, manhã, tarde, noite) |
| ocorrencias | `dia_semana_num`, `fim_de_semana` | ordem do dia (1 = segunda) e dia útil/fim de semana |
| ocorrencias | `gravidade`, `com_morte` | pior desfecho do acidente e indicador 0/1 de morte |
| ocorrencias | `br_nome`, `trecho_10km` | BR formatada e trecho de 10 km |
| ocorrencias | `lat_long`, `cod_ibge` | coordenada para mapa e código IBGE do município |
| pessoas | `faixa_etaria`, `grupo_veiculo`, `idade_veiculo` | faixa de idade, veículo agrupado, idade do veículo |
| pessoas | `vitima_fatal`, `ferido` | indicadores 0/1 |
| municipios | `acidentes_100mil_hab`, `mortos_100mil_hab`, `letalidade_pct`, `porte` | indicadores relativos à população |

**Relacionamentos para o Looker Studio:** `ocorrencias.id = pessoas.id` e `ocorrencias.cod_ibge = municipios.cod_ibge`.